## Test de l'API de scoring Home Credit

Ce notebook permet de :

1. Charger un échantillon de clients depuis le jeu de test.
2. Vérifier que l'API est bien démarrée (`/health`).
3. Envoyer un client à l’endpoint `/predict`.
4. Inspecter la probabilité de défaut et la décision renvoyées par l’API.


## Sommaire

[1. Imports & configuration de base](#1-imports--configuration-de-base)

[2. Chargement du dataset de test et sélection d'un client](#2-chargement-du-dataset-de-test-et-sélection-dun-client)

[3. Vérifier que lAPI répond bien sur health](#3-vérifier-que-lapi-répond-bien-sur-health)

[4. Préparation du payload à envoyer à predict](#4-préparation-du-payload-à-envoyer-à-predict)

[5. Appel à lendpoint predict](#5-appel-à-lendpoint-predict)

[6. Résumé lisible de la prédiction](#6-résumé-lisible-de-la-prédiction)



## 1. Imports & configuration de base

In [1]:
import os
import math
import requests
import pandas as pd
import json
import numpy as np

from dotenv import load_dotenv

In [2]:
# Chargement des variables d'environnement (.env à la racine du projet)
load_dotenv()

# URL de base de l'API (par défaut : FastAPI en local via uvicorn)
# Exemple de lancement : uvicorn src.api.app:app --reload
# BASE_URL = os.getenv("API_URL", "http://127.0.0.1:8000")
BASE_URL = "http://localhost:8000"

print("BASE_URL =", BASE_URL)

BASE_URL = http://localhost:8000


## 2. Chargement du dataset de test et sélection d'un client

In [3]:
# Charger la liste officielle des features attendues par le modèle
with open("../models/feature_names.json", "r") as f:
    feature_names = json.load(f)

len(feature_names)

85

In [4]:
TEST_PATH = "../data/processed/df_test.csv"

df_test = pd.read_csv(TEST_PATH)
print("Shape df_test :", df_test.shape)


Shape df_test : (48744, 817)


In [5]:
df_test = df_test.drop(columns=["TARGET"], errors="ignore")

In [6]:
df_test = df_test.reindex(columns=feature_names, fill_value=0)

In [7]:
# Reindex pour forcer la même structure que le modèle
df_test_aligned = df_test.reindex(columns=feature_names, fill_value=0)

df_test_aligned.shape

(48744, 85)

In [8]:
sample_raw = df_test_aligned.iloc[0]
sample_raw.head()

EXT_SOURCE_3      0.15952
EXT_SOURCE_2     0.789654
EXT_SOURCE_1     0.752614
PAYMENT_RATE     0.036147
DAYS_EMPLOYED     -2329.0
Name: 0, dtype: object

In [9]:
# Nettoyage des valeurs qui ne passent pas en JSON
sample_clean = sample_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

In [10]:
sample_clean = sample_raw.copy()
sample_clean = sample_clean.replace([np.inf, -np.inf], np.nan)
sample_clean = sample_clean.fillna(0)

## 3. Vérifier que l'API répond bien sur /health

In [16]:
health = requests.get(f"{BASE_URL}/health")
print("Status code /health :", health.status_code)
print("Réponse /health :")
health.json()

Status code /health : 200
Réponse /health :


{'status': 'ok', 'model_loaded': True, 'n_features': 85}

## 4. Préparation du payload à envoyer à /predict

In [14]:
payload = {"data": sample_clean.to_dict()}
payload

{'data': {'EXT_SOURCE_3': 0.1595195404777181,
  'EXT_SOURCE_2': 0.7896543511176771,
  'EXT_SOURCE_1': 0.7526144906031748,
  'PAYMENT_RATE': 0.0361471518987341,
  'DAYS_EMPLOYED': -2329.0,
  'AMT_ANNUITY': 20560.5,
  'INSTAL_DPD_MEAN': 1.5714285714285714,
  'PREV_CNT_PAYMENT_MEAN': 8.0,
  'INSTAL_AMT_PAYMENT_MIN': 3951.0,
  'ACTIVE_DAYS_CREDIT_MAX': -49.0,
  'INSTAL_DAYS_ENTRY_PAYMENT_MEAN': -2195.0,
  'PREV_NAME_CONTRACT_STATUS_Refused_MEAN': 0.0,
  'INSTAL_AMT_PAYMENT_SUM': 41195.925,
  'DAYS_ID_PUBLISH': -812,
  'ANNUITY_INCOME_PERC': 0.1523,
  'NAME_EDUCATION_TYPE_Higher_education': 0,
  'BURO_DAYS_CREDIT_MEAN': -735.0,
  'POS_MONTHS_BALANCE_SIZE': 9.0,
  'BURO_AMT_CREDIT_MAX_OVERDUE_MEAN': 0,
  'DAYS_REGISTRATION': -5170.0,
  'PREV_APP_CREDIT_PERC_MEAN': 1.0440786984487325,
  'ACTIVE_DAYS_CREDIT_ENDDATE_MIN': 411.0,
  'BURO_AMT_CREDIT_SUM_DEBT_MEAN': 85240.92857142857,
  'INSTAL_DAYS_ENTRY_PAYMENT_SUM': -15365.0,
  'INSTAL_PAYMENT_PERC_MEAN': 1.0,
  'INCOME_CREDIT_PERC': 0.23734177

## 5. Appel à l'endpoint /predict

In [18]:
API_URL = "http://localhost:8000/predict_proba"

response = requests.post(API_URL, json=payload)

result = response.json()
result


{'probability_default': 0.03232002171682745,
 'decision': 0,
 'threshold_used': 0.27}

## 6. Résumé lisible de la prédiction


In [20]:
if response.status_code == 200:
    proba = result["probability_default"]
    pred = result["decision"]
    thr = result["threshold_used"]

    decision_text = (
        "REFUS — risque de défaut élevé"
        if pred == 1
        else "ACCEPTATION — risque de défaut faible"
    )

    print(f"Probabilité de défaut : {proba:.4f}")
    print(f"Seuil métier utilisé  : {thr:.4f}")
    print(f"Classe prédite        : {pred}  →  {decision_text}")

else:
    print("Erreur retournée par l'API :")
    print(result)


Probabilité de défaut : 0.0323
Seuil métier utilisé  : 0.2700
Classe prédite        : 0  →  ACCEPTATION — risque de défaut faible
